In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
import warnings

warnings.filterwarnings("ignore")


In [2]:
features_df = pd.read_csv("data/features.csv")
targets_df = pd.read_csv("data/targets.csv")

In [3]:
X = features_df.values
y = targets_df.values.ravel()

In [4]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)

In [6]:
print(f"\nTrain-Test Split (70-30 stratified):")
print(f"   Training set: {X_train.shape[0]} samples")
print(f"   Test set: {X_test.shape[0]} samples")


Train-Test Split (70-30 stratified):
   Training set: 105 samples
   Test set: 45 samples


In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
print(f"\nFeature Scaling Applied (StandardScaler)")
print(f"   Mean: {scaler.mean_}")
print(f"   Std: {scaler.scale_}")



Feature Scaling Applied (StandardScaler)
   Mean: [5.87333333 3.05047619 3.78571429 1.2047619 ]
   Std: [0.85882164 0.4519005  1.77428341 0.77513513]


In [11]:
classifiers = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Support Vector Machine": SVC(random_state=42),
}
baseline_results = {}

In [12]:
print("\nTraining baseline models...\n")
for name, clf in classifiers.items():
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_test_scaled)

    cv_scores = cross_val_score(clf, X_train_scaled, y_train, cv=5)

    baseline_results[name] = {
        "model": clf,
        "y_pred": y_pred,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted"),
        "recall": recall_score(y_test, y_pred, average="weighted"),
        "f1": f1_score(y_test, y_pred, average="weighted"),
        "cv_mean": cv_scores.mean(),
        "cv_std": cv_scores.std(),
        "confusion_matrix": confusion_matrix(y_test, y_pred),
    }

    print(f"[OK] {name}")
    print(f"  Accuracy: {baseline_results[name]['accuracy']:.4f}")
    print(f"  F1-Score: {baseline_results[name]['f1']:.4f}")
    print(
        f"  CV Score: {baseline_results[name]['cv_mean']:.4f} (+/-{baseline_results[name]['cv_std']:.4f})"
    )


Training baseline models...

[OK] Logistic Regression
  Accuracy: 0.9111
  F1-Score: 0.9107
  CV Score: 0.9810 (+/-0.0233)
[OK] Decision Tree
  Accuracy: 0.9111
  F1-Score: 0.9107
  CV Score: 0.9429 (+/-0.0190)
[OK] Random Forest
  Accuracy: 0.8889
  F1-Score: 0.8878
  CV Score: 0.9524 (+/-0.0301)
[OK] Support Vector Machine
  Accuracy: 0.9333
  F1-Score: 0.9333
  CV Score: 0.9714 (+/-0.0233)


Time to tune hyper parameters for each models to squeeze out extra accuracy

In [13]:
param_grids = {
    "Logistic Regression": {
        "C": [0.01, 0.1, 1, 10, 100],
        "penalty": ["l2"],
        "solver": ["lbfgs", "liblinear"],
    },
    "Decision Tree": {
        "max_depth": [3, 5, 7, 10, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "criterion": ["gini", "entropy"],
    },
    "Random Forest": {
        "n_estimators": [50, 100, 200],
        "max_depth": [5, 10, 15, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
    },
    "Support Vector Machine": {
        "C": [0.1, 1, 10, 100],
        "kernel": ["linear", "rbf", "poly"],
        "gamma": ["scale", "auto"],
    },
}

tuned_results = {}

In [14]:
print("\nPerforming GridSearchCV with 5-fold cross-validation...\n")

for name, clf in classifiers.items():
    print(f"[TUNING] {name}...")

    grid_search = GridSearchCV(
        clf, param_grids[name], cv=5, scoring="accuracy", n_jobs=-1, verbose=0
    )
    grid_search.fit(X_train_scaled, y_train)

    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test_scaled)

    cv_scores = cross_val_score(best_model, X_train_scaled, y_train, cv=5)

    tuned_results[name] = {
        "model": best_model,
        "y_pred": y_pred,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted"),
        "recall": recall_score(y_test, y_pred, average="weighted"),
        "f1": f1_score(y_test, y_pred, average="weighted"),
        "cv_mean": cv_scores.mean(),
        "cv_std": cv_scores.std(),
        "confusion_matrix": confusion_matrix(y_test, y_pred),
        "best_params": grid_search.best_params_,
        "best_cv_score": grid_search.best_score_,
    }

    print(f"  [OK] Best CV Score: {tuned_results[name]['best_cv_score']:.4f}")
    print(f"  [OK] Test Accuracy: {tuned_results[name]['accuracy']:.4f}")
    print(f"  [OK] Best Parameters: {tuned_results[name]['best_params']}")

    print()

print("Tuning complete.\n")



Performing GridSearchCV with 5-fold cross-validation...

[TUNING] Logistic Regression...
  [OK] Best CV Score: 0.9810
  [OK] Test Accuracy: 0.9111
  [OK] Best Parameters: {'C': 1, 'penalty': 'l2', 'solver': 'lbfgs'}

[TUNING] Decision Tree...
  [OK] Best CV Score: 0.9524
  [OK] Test Accuracy: 0.9778
  [OK] Best Parameters: {'criterion': 'gini', 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}

[TUNING] Random Forest...
  [OK] Best CV Score: 0.9619
  [OK] Test Accuracy: 0.9111
  [OK] Best Parameters: {'max_depth': 5, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 100}

[TUNING] Support Vector Machine...
  [OK] Best CV Score: 0.9810
  [OK] Test Accuracy: 0.9333
  [OK] Best Parameters: {'C': 100, 'gamma': 'scale', 'kernel': 'linear'}

Tuning complete.



Let's compare the results and announce the winner model

In [15]:
comparison_data = []
for name in classifiers.keys():
    comparison_data.append(
        {
            "Model": name,
            "Type": "Baseline",
            "Accuracy": baseline_results[name]["accuracy"],
            "Precision": baseline_results[name]["precision"],
            "Recall": baseline_results[name]["recall"],
            "F1-Score": baseline_results[name]["f1"],
            "CV Mean": baseline_results[name]["cv_mean"],
            "CV Std": baseline_results[name]["cv_std"],
        }
    )
    comparison_data.append(
        {
            "Model": name,
            "Type": "Tuned",
            "Accuracy": tuned_results[name]["accuracy"],
            "Precision": tuned_results[name]["precision"],
            "Recall": tuned_results[name]["recall"],
            "F1-Score": tuned_results[name]["f1"],
            "CV Mean": tuned_results[name]["cv_mean"],
            "CV Std": tuned_results[name]["cv_std"],
        }
    )

comparison_df = pd.DataFrame(comparison_data)
print("\nBaseline vs Tuned Performance:\n")
print(comparison_df.to_string(index=False))


Baseline vs Tuned Performance:

                 Model     Type  Accuracy  Precision   Recall  F1-Score  CV Mean   CV Std
   Logistic Regression Baseline  0.911111   0.915535 0.911111  0.910714 0.980952 0.023328
   Logistic Regression    Tuned  0.911111   0.915535 0.911111  0.910714 0.980952 0.023328
         Decision Tree Baseline  0.911111   0.915535 0.911111  0.910714 0.942857 0.019048
         Decision Tree    Tuned  0.977778   0.979167 0.977778  0.977753 0.952381 0.030117
         Random Forest Baseline  0.888889   0.898148 0.888889  0.887767 0.952381 0.030117
         Random Forest    Tuned  0.911111   0.915535 0.911111  0.910714 0.961905 0.035635
Support Vector Machine Baseline  0.933333   0.934524 0.933333  0.933259 0.971429 0.023328
Support Vector Machine    Tuned  0.933333   0.944444 0.933333  0.932660 0.980952 0.023328


In [17]:
improvements = []
for name in classifiers.keys():
    baseline_acc = baseline_results[name]["accuracy"]
    tuned_acc = tuned_results[name]["accuracy"]
    improvement = ((tuned_acc - baseline_acc) / baseline_acc) * 100
    improvements.append(
        {
            "Model": name,
            "Baseline Accuracy": baseline_acc,
            "Tuned Accuracy": tuned_acc,
            "Improvement (%)": improvement,
        }
    )

improvement_df = pd.DataFrame(improvements)
print("\n\nPerformance Improvements:\n")
print(improvement_df.to_string(index=False))



Performance Improvements:

                 Model  Baseline Accuracy  Tuned Accuracy  Improvement (%)
   Logistic Regression           0.911111        0.911111         0.000000
         Decision Tree           0.911111        0.977778         7.317073
         Random Forest           0.888889        0.911111         2.500000
Support Vector Machine           0.933333        0.933333         0.000000


In [19]:
for name in classifiers.keys():
    print(f"\n{'=' * 50}")
    print(f"{name.upper()} - TUNED MODEL")
    print(f"{'=' * 50}")
    print("\nClassification Report:")
    print(
        classification_report(
            y_test, tuned_results[name]["y_pred"], target_names=label_encoder.classes_
        )
    )


LOGISTIC REGRESSION - TUNED MODEL

Classification Report:
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        15
Iris-versicolor       0.82      0.93      0.88        15
 Iris-virginica       0.92      0.80      0.86        15

       accuracy                           0.91        45
      macro avg       0.92      0.91      0.91        45
   weighted avg       0.92      0.91      0.91        45


DECISION TREE - TUNED MODEL

Classification Report:
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        15
Iris-versicolor       1.00      0.93      0.97        15
 Iris-virginica       0.94      1.00      0.97        15

       accuracy                           0.98        45
      macro avg       0.98      0.98      0.98        45
   weighted avg       0.98      0.98      0.98        45


RANDOM FOREST - TUNED MODEL

Classification Report:
                 precision    r

In [21]:
print("Saving results and generating visualizations...")
print("=" * 100)

results_summary = {
    "baseline_results": baseline_results,
    "tuned_results": tuned_results,
    "comparison_df": comparison_df,
    "improvement_df": improvement_df,
    "label_encoder": label_encoder,
    "scaler": scaler,
    "X_test": X_test_scaled,
    "y_test": y_test,
}


Saving results and generating visualizations...


In [22]:
import pickle

with open("ml_pipeline_results.pkl", "wb") as f:
    pickle.dump(results_summary, f)

print("\nResults saved to: ml_pipeline_results.pkl")
print("Ready for data vis pipeline...")



Results saved to: ml_pipeline_results.pkl
Ready for data vis pipeline...
